In [1]:
import torch
from torch import nn
from d2l import torch as d2l

In [ ]:
def batch_norm(X,gamma,beta,moving_mean,moving_var,eps,momentum):
    if not torch.is_grad_enabled():
        X_hat = (X-moving_mean)/ torch.sqrt(moving_var+eps)  #在推理模式下，直接采取训练中留下的全局均值和方差
    else:
        assert len(X.shape) in (2,4)
        if len(X.shape) == 2:  # [batch_size, features]
            mean = X.mean(dim=0)  #在批量维度上求均值
            var = ((X - mean) ** 2).mean(dim=0)
        else:
            mean = X.mean(dim = (0,2,3),keepdim = True)
            var = ((X-mean)**2).mean(dim=(0, 2, 3), keepdim=True)

        X_hat = (X - mean) / torch.sqrt(var + eps)
        moving_mean = momentum * moving_mean + (1.0 - momentum) * mean
        moving_var = momentum * moving_var + (1.0 - momentum) * var
    Y = gamma * X_hat + beta  # 缩放和移位
    return Y, moving_mean.data, moving_var.data

moving_mean (旧值)：截至上一个 Batch 为止，积累下来的全局均值。mean (当前值)：当前这个 Batch 计算出的均值。momentum (动量)：一个通常设为 $0.9$ 或 $0.99$ 的系数，决定了“保留多少旧记忆”。这样，在预测时候及时用户只输入一个图片也可以计算出方差

In [ ]:
class BatchNorm(nn.Module):
    def __init__(self, num_features, num_dims):
        super().__init__
        if(num_dims) == 2:
            shape = (1,num_features) 
        else:
            shape = (1, num_features, 1, 1)

        self.gamma = nn.Parameter(torch.ones(shape)) #pytorch会自动优化
        self.beta = nn.Parameter(torch.zeros(shape))

        self.moving_mean = torch.zeros(shape)
        self.moving_var = torch.ones(shape)

    def forward(self, X):
        if self.moving_mean.device != X.device:
            self.moving_mean = self.moving_mean.to(X.device)
            self.moving_var = self.moving_var.to(X.device)
        # 保存更新过的moving_mean和moving_var
        Y, self.moving_mean, self.moving_var = batch_norm(
            X, self.gamma, self.beta, self.moving_mean,
            self.moving_var, eps=1e-5, momentum=0.9)
        return Y